In [ ]:
import os
import re
from getpass import getpass

# 1. CONFIGURACIÓN DE LA API DE GROQ
print("=== CONFIGURACIÓN DEL ENTORNO ===")
if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("Introduce tu GROQ API Key (obtenla en console.groq.com): ")
print("✅ API Key configurada correctamente.\n")

# 2. ESCUDO DE SEGURIDAD (PII) - Heredado de tu v4.1
def analizar_seguridad(texto):
    """
    Detecta patrones PII, los anonimiza y devuelve el texto limpio 
    junto con el reporte de hallazgos.
    """
    patrones = {
        "DNI": r"\b\d{8}[A-Z]\b",
        "Teléfono": r"\b[6789]\d{8}\b",
        "Matrícula": r"\b\d{4}[A-Z]{3}\b",
        "IBAN": r"\bES\d{22}\b"
    }
    
    hallazgos = []
    texto_limpio = texto
    
    for tipo, regex in patrones.items():
        matches = re.findall(regex, texto_limpio, re.IGNORECASE)
        for m in matches:
            anonimo = m[:2] + "*" * (len(m)-4) + m[-2:]
            hallazgos.append({"tipo": tipo, "anonimo": anonimo})
            texto_limpio = texto_limpio.replace(m, f"[{tipo} PROTEGIDO]")
            
    return hallazgos, texto_limpio

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

class SegurPlusLLM:
    """Chatbot con LLM (Groq), ventana de memoria de 5 turnos e integración RAG simulada."""
    
    def __init__(self, tipo_incidencia, num_poliza):
        # Inicializamos el LLM de Groq
        self.llm = ChatGroq(
            model_name="llama-3.3-70b-versatile",
            temperature=0.3
        )
        
        # Historial limitado a 5 turnos
        self.history = []
        self.max_history = 5 
        self.tipo_incidencia = tipo_incidencia
        self.num_poliza = num_poliza
        
        # --- BASE DE CONOCIMIENTO (RAG SIMULADO) ---
        self.knowledge_base = {
            "Coche": "[RAG INFO] Las grúas solo se envían si el coche está inmovilizado en vía pública. Los taxis cubren un máximo de 50km. No se cubren pinchazos simples si el coche tiene rueda de repuesto.",
            "Hogar": "[RAG INFO] Se envía técnico de urgencia (fontanero/electricista) solo para siniestros graves que afecten la habitabilidad de la vivienda. Daños estéticos no urgentes están excluidos de asistencia inmediata.",
            "Gestiones Administrativas": "[RAG INFO] Para cambios de datos bancarios o IBAN es obligatorio que el cliente rellene el Formulario de Modificación de Datos y lo envíe firmado por email."
        }
        
    def respond(self, user_input: str) -> str:
        # 1. RETRIEVAL (RAG)
        contexto_rag = self.knowledge_base.get(self.tipo_incidencia, "No hay información específica disponible.")
        
        # System Prompt
        system_prompt = f"""Eres un agente de asistencia de SegurPlus.
        Estás atendiendo una incidencia de tipo: {self.tipo_incidencia.upper()}.
        El número de póliza validado del cliente es: {self.num_poliza}.
        
        CONTRATO Y REGLAS CORPORATIVAS (Recuperadas vía RAG):
        {contexto_rag}
        
        TU OBJETIVO: Determinar la solución adecuada basada en lo que te cuente el cliente y respetando el CONTRATO RAG anterior.
        Tus opciones de resolución son:
        - Mandar un técnico a casa.
        - Mandar una grúa.
        - Llamar a una ambulancia.
        - Pedir un taxi.
        - Pasar con un agente humano (para quejas o dudas complejas).
        
        REGLAS:
        1. Sé empático, profesional y directo.
        2. Haz solo una pregunta a la vez si necesitas más datos.
        3. Cuando sepas qué hacer, dile al cliente la acción exacta que vas a tomar de la lista de opciones basándote en lo permitido por el contrato RAG.
        4. NUNCA pidas DNI, teléfonos o IBAN."""
        
        messages = [SystemMessage(content=system_prompt)]
        
        recent_history = self.history[-self.max_history:]
        for human_msg, ai_msg in recent_history:
            messages.append(HumanMessage(content=human_msg))
            messages.append(AIMessage(content=ai_msg))
            
        messages.append(HumanMessage(content=user_input))
        
        response = self.llm.invoke(messages)
        ai_response = response.content
        
        self.history.append((user_input, ai_response))
        
        return ai_response

In [ ]:
def iniciar_chatbot_segurplus():
    print("\n" + "="*50)
    print("🤖 SISTEMA SEGURPLUS v5.0 (HÍBRIDO LLM con RAG)")
    print("="*50)
    
    estado_actual = "CONTEXTO"
    tipo_incidencia = None
    num_poliza = None
    bot_llm = None
    
    print("\nChatbot: ¡Hola! Bienvenido a SegurPlus. Para derivar tu consulta correctamente, por favor elige una opción:")
    print("  1. Seguro de Coche")
    print("  2. Seguro de Hogar")
    print("  3. Modificación de Datos / Gestiones")

    while True:
        usuario = input("\nTú: ").strip()
        
        if usuario.lower() in ['salir', 'exit', 'quit']:
            print("Chatbot: Gracias por usar SegurPlus. ¡Hasta pronto!")
            break
            
        if not usuario:
            continue

        alerta_datos, texto_seguro = analizar_seguridad(usuario)
        if alerta_datos:
            print("\n🛡️ [REPORTE DE PRIVACIDAD]")
            for dato in alerta_datos:
                print(f"  - He ocultado el dato tipo {dato['tipo']} ({dato['anonimo']}) por tu seguridad.")

        if estado_actual == "CONTEXTO":
            if "1" in texto_seguro or "coche" in texto_seguro.lower():
                tipo_incidencia = "Coche"
                estado_actual = "POLIZA"
                print("\nChatbot: Entendido, asistencia para Coche. Por favor, indícame tu número de póliza (8 dígitos).")
            elif "2" in texto_seguro or "hogar" in texto_seguro.lower():
                tipo_incidencia = "Hogar"
                estado_actual = "POLIZA"
                print("\nChatbot: Entendido, asistencia de Hogar. Por favor, indícame tu número de póliza (8 dígitos).")
            elif "3" in texto_seguro or "datos" in texto_seguro.lower():
                tipo_incidencia = "Gestiones Administrativas"
                estado_actual = "POLIZA"
                print("\nChatbot: Entendido, área de Gestiones. Por favor, indícame tu número de póliza (8 dígitos).")
            else:
                print("\nChatbot: Por favor, responde con 1, 2 o 3 para poder ayudarte mejor.")
                
        elif estado_actual == "POLIZA":
            match_poliza = re.search(r"\b\d{8}\b", texto_seguro)
            if match_poliza:
                num_poliza = match_poliza.group(0)
                estado_actual = "LLM_CHAT"
                bot_llm = SegurPlusLLM(tipo_incidencia, num_poliza)
                print(f"\nChatbot: Póliza {num_poliza} validada correctamente. Cuéntame, ¿qué ha ocurrido?")
            else:
                print("\nChatbot: No detecto un formato válido. Recuerda que el número de póliza debe tener exactamente 8 números.")
                
        elif estado_actual == "LLM_CHAT":
            respuesta_llm = bot_llm.respond(texto_seguro)
            print(f"\nChatbot: {respuesta_llm}")

# Ejecutar la aplicación
iniciar_chatbot_segurplus()